In [ ]:
import sys
sys.path.append('..')
import torch
import cying.nn as cynn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
device = torch.device('cuda:0')

size = 256
transform = transforms.Compose([
    transforms.Resize(size),
    transforms.CenterCrop(size),
    transforms.ToTensor()
])

train_dataset = datasets.ImageFolder(
    root='/root/autodl-tmp/datasets/ImageNet/train', 
    transform=transform
)

train_loader = DataLoader(
    train_dataset, 
    batch_size=128, 
    shuffle=True, 
    num_workers=4, 
    pin_memory=True
)
valid_dataset = datasets.ImageFolder(
    root='/root/autodl-tmp/datasets/ImageNet/valid', 
    transform=transform
)

valid_loader = DataLoader(
    valid_dataset, 
    batch_size=128, 
    shuffle=True, 
    num_workers=4, 
    pin_memory=True
)

test_dataset = datasets.ImageFolder(
    root='/root/autodl-tmp/datasets/ImageNet/test', 
    transform=transform
)

test_loader = DataLoader(
    test_dataset, 
    batch_size=128, 
    shuffle=True, 
    num_workers=4, 
    pin_memory=True
)

from torchinfo import summary
params = [
    {
        'size': (256, 256),
        'in_channels': 3,
        'out_channels': 8,
        'hidden_width': 128,
        'spe_opt_size': (16,16),
        'spa_opt_size': 11
    },
    {
        'size': (128, 128),
        'in_channels': 8,
        'out_channels': 32,
        'hidden_width': 128,
        'spe_opt_size': (16,16),
        'spa_opt_size': 7
    },
    {
        'size': (32, 32),
        'in_channels': 32,
        'out_channels': 64,
        'hidden_width': 128,
        'spe_opt_size': (8,8),
        'spa_opt_size': 5
    },
    {
        'size': (8, 8),
        'in_channels': 64,
        'out_channels': 128,
        'hidden_width': 256,
        'spe_opt_size': (4,4),
        'spa_opt_size': 3
    }
]
class ImageClassifier(cynn.OperatorModel2d):
    def __init__(self, params, num_classes=1000):
        super().__init__(params)
        self.classifier = torch.nn.Linear(params[-1]['out_channels'], num_classes)

    def forward(self, x):
        x = super().forward(x)
        x = torch.mean(x, dim=(-2, -1))
        x = self.classifier(x)
        return x
classifier = ImageClassifier(
    params,
    num_classes=1000
).to(device)
summary(classifier)

from tqdm import tqdm
epochs = 10
lr = 1e-3

optim_train = torch.optim.Adam(classifier.parameters(), lr=lr)
optim_valid = torch.optim.Adam(classifier.get_opt_weight(), lr=lr)

loss_fn = torch.nn.CrossEntropyLoss()

for i in range(epochs):
    classifier.train()
    train_loss = 0.0
    for images, labels in tqdm(train_loader, desc=f"Epoch {i+1}/{epochs}"):
        optim_train.zero_grad()

        outputs = classifier(images.to(device))
        loss = loss_fn(outputs, labels.to(device))
        loss.backward()

        optim_train.step()

        train_loss += loss.item() * images.size(0)

    valid_loss = 0.0
    for images, labels in tqdm(valid_loader, desc=f"Epoch {i+1}/{epochs}"):
        optim_valid.zero_grad()

        outputs = classifier(images.to(device))
        loss = loss_fn(outputs, labels.to(device))
        loss.backward()

        optim_valid.step()
        
        valid_loss += loss.item() * images.size(0)

    classifier.eval()
    torch.save(classifier, './imagenet_classifier.pth')
    test_loss = 0.0
    for images, labels in tqdm(test_loader, desc=f"Epoch {i+1}/{epochs}"):
        outputs = classifier(images.to(device))
        loss = loss_fn(outputs, labels.to(device))
        test_loss += loss.item() * images.size(0)
    

    train_loss /= len(train_loader.dataset)
    valid_loss /= len(valid_loader.dataset)
    test_loss /= len(test_loader.dataset)
    print(f"Epoch {i+1}/{epochs}, Training Loss: {train_loss:.4f}, Validation Loss: {valid_loss:.4f}, Test Loss: {test_loss:.4f}")